In [0]:
create or replace temporary view jess_file as 
WITH npi_mapping AS (
    SELECT
        current_npi,
        current_name,
        mapped_npi,
        mapped_name
    FROM (
        VALUES
            ('1144211301', 'Atrium Health Wake Forest Baptist Medical Center', '1295789907', 'Atrium Health'),
            ('1184779332', 'Childrens Healthcare Of Atlanta Scottish Rite Hospital', '1235339227', 'Emory University Hospital'),
            ('1851458038', 'Dr Patrick Leavey MD Office', '1235582925', 'UT Health'),
            ('1225259039', 'Greenwood Genetics Center Inc.', '1649261462', 'Greenwood Genetics Center Inc.'),
            ('1831318856', 'Greenwood Genetics Center Inc.', '1649261462', 'Greenwood Genetics Center Inc.'),
            ('1356496772', 'Kaiser Permanente Fontana Medical Center', '1013062769', 'Kaiser Permanente San Diego Medical Center'),
            ('1003947599', 'Univ. Pediatric Associates Inc.', '1144266024', 'Indiana University Health'),
            ('1205822236', 'Yale Medicine', '1013924182', 'Yale-New Haven Hospital')
    ) AS t(current_npi, current_name, mapped_npi, mapped_name)
)
select * from npi_mapping

In [0]:
with mapping as (
  select a.mapped_npi, a.mapped_name, b.primary_npi, b.primary_name
from jess_file as a
left join com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi as b
on a.mapped_npi = b.secondary_npi
)
select distinct coalesce(primary_npi, mapped_npi) as hco_npi
from mapping

In [0]:
select distinct *
from com_edp_prd.cmpa_insights_internal_schema.reference_file_12_24_2025
where hco_npi in (with mapping as (
  select a.mapped_npi, a.mapped_name, b.primary_npi, b.primary_name
from jess_file as a
left join com_edp_prd.cmpa_insights_internal_schema.secondary_to_primary_npi as b
on a.mapped_npi = b.secondary_npi
)
select distinct coalesce(primary_npi, mapped_npi) as hco_npi
from mapping)

In [0]:
with t1 as (select distinct *, row_number() over(partition by hco_npi order by hcp_npi asc) as rn
from com_edp_prd.cmpa_insights_internal_schema.reference_file_12_12_2025
where hco_npi in (select distinct current_npi from jess_file))
select *
from t1 
where rn <= 4
order by hco_npi